# 動画×再生リスト対応表（単発抽出）

対象チャンネル: `UCGMG8BNfA8gsH9Rn_d_yW2A`

動画ごとにどの再生リストに紐づいているかを一度きり調べるための独立ノートブック。
他のnotebookや `sixfonia_analytics` パッケージには依存しない。

- 対象は公開再生リストのみ（APIキー認証、OAuth不要）
- 出力はlong形式（1動画が複数再生リストに属する場合は行が増える。どの再生リストにも属さない動画は `playlist_id`/`playlist_name` が空欄の1行になる）
- タイトルにカンマが含まれても列がずれないよう、出力はTSV（タブ区切り）
- 実行前に Colab の Secrets に `YOUTUBE_API_KEY` を設定しておくこと

## 1. セットアップ

In [ ]:
!pip install -q google-api-python-client

from googleapiclient.discovery import build
from google.colab import userdata
import pandas as pd

## 2. 設定・認証

In [ ]:
CHANNEL_ID = "UCGMG8BNfA8gsH9Rn_d_yW2A"

API_KEY = userdata.get("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

## 3. チャンネルの全動画リスト取得（uploads再生リスト経由）

プレイリスト未所属の動画も含めた「マスター動画リスト」として使う。

In [ ]:
channel_res = youtube.channels().list(
    part="contentDetails",
    id=CHANNEL_ID,
).execute()
uploads_playlist_id = channel_res["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

video_rows = []
page_token = None
while True:
    res = youtube.playlistItems().list(
        part="snippet",
        playlistId=uploads_playlist_id,
        maxResults=50,
        pageToken=page_token,
    ).execute()
    for item in res["items"]:
        snippet = item["snippet"]
        video_rows.append({
            "video_id": snippet["resourceId"]["videoId"],
            "video_title": snippet["title"],
        })
    page_token = res.get("nextPageToken")
    if not page_token:
        break

df_videos = pd.DataFrame(video_rows).drop_duplicates(subset="video_id")
print(f"total videos: {len(df_videos)}")

## 4. チャンネルの全再生リスト取得

In [ ]:
playlists = []
page_token = None
while True:
    res = youtube.playlists().list(
        part="snippet",
        channelId=CHANNEL_ID,
        maxResults=50,
        pageToken=page_token,
    ).execute()
    for item in res["items"]:
        playlists.append({
            "playlist_id": item["id"],
            "playlist_title": item["snippet"]["title"],
        })
    page_token = res.get("nextPageToken")
    if not page_token:
        break

print(f"total playlists: {len(playlists)}")

## 5. 各再生リストの所属動画取得

In [ ]:
membership_rows = []
for playlist in playlists:
    page_token = None
    while True:
        res = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist["playlist_id"],
            maxResults=50,
            pageToken=page_token,
        ).execute()
        for item in res["items"]:
            membership_rows.append({
                "video_id": item["contentDetails"]["videoId"],
                "playlist_id": playlist["playlist_id"],
                "playlist_title": playlist["playlist_title"],
            })
        page_token = res.get("nextPageToken")
        if not page_token:
            break

df_membership = pd.DataFrame(membership_rows)
print(f"total (video, playlist) pairs: {len(df_membership)}")

## 6. long形式への統合（未所属動画も1行として含める）

In [ ]:
df_result = df_videos.merge(df_membership, on="video_id", how="left")
df_result = df_result.rename(columns={"playlist_title": "playlist_name"})
df_result = df_result[["video_id", "video_title", "playlist_id", "playlist_name"]]
df_result = df_result.sort_values(["video_id", "playlist_id"]).reset_index(drop=True)
df_result

## 7. 出力（TSVダウンロード）

In [ ]:
OUTPUT_PATH = "video_playlist_map.tsv"
df_result.to_csv(OUTPUT_PATH, index=False, sep="\t", encoding="utf-8-sig")

from google.colab import files
files.download(OUTPUT_PATH)